# Fraud Investigation Task, 1-  Data Enrichment 
Author: Maha Abdelshafy
 
Date: 2025  

This notebook covers:
1. Data loading & initial exploration  
2. Data Engineering  




## **Section 1:Technical Data Preparation & Enrichment**


- "This section prepares the raw dataset for fraud analysis by performing data cleaning, structural normalization, IP expansion, GeoIP enrichment, and User-Agent parsing. 
- These technical transformations ensure that the dataset is consistent, traceable, and analytically ready for cross-column integrity checks and fraud-pattern extraction."

---------------------

###  **1) Data Loading & Basic Overview**

In [2]:
#import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ipaddress import ip_address
from sklearn.preprocessing import LabelEncoder
import json
import requests
import networkx
import re
import socket
import ipaddress
import geoip2.database
sns.set(style='whitegrid')
# Set pandas display options for easier debugging and exploration
from IPython.display import display
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

In [3]:
#load the Dataset
df=pd.read_excel('test.xlsx')
df.head(3)


,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os,browser,ips
0,00993a56-f44b-448d-9d33-78c16f0be636,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac OS X 10.15.1,Chrome 78.0.3904,185.97.201.89
1,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"64.52.83.86,154.160.9.113,64.52.83.10"
2,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,154.160.10.201


In [4]:
#Basic Overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1436 entries, 0 to 1435
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   device_id           1436 non-null   object
 1   identity            1436 non-null   object
 2   bank                1436 non-null   object
 3   device_fingerprint  1436 non-null   object
 4   gpu_renderers       1436 non-null   object
 5   screen              1436 non-null   object
 6   os                  1436 non-null   object
 7   browser             1436 non-null   object
 8   ips                 1436 non-null   object
dtypes: object(9)
memory usage: 101.1+ KB


In [5]:
# Unique values per column
df.nunique()

device_id             169
identity              333
bank                   11
device_fingerprint     36
gpu_renderers          28
screen                 27
os                     24
browser                36
ips                   830
dtype: int64

In [6]:
df.describe(include='all')

,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os,browser,ips
count,1436,1436,1436,1436,1436,1436,1436,1436,1436
unique,169,333,11,36,28,27,24,36,830
top,614d661d-3a4d-47db-9e2b-874e481260ea,2577384,Bank8,C45D75CA94FD92796B0F51000CF754B035AA4E48,-,"(780, 360, 24)",Android,Chrome 78.0.3904,185.97.201.65
freq,391,223,400,391,658,416,450,290,50


**Insights**

- The dataset loads correctly and shows no critical structural defects.  
- Several columns contain non-uniform textual patterns, indicating that normalization will be necessary before enrichment steps.   
- The dataset is in a workable state for quality checks, feature extraction, and graph-based linking in upcoming sections.





## **2) : Data quality checks**

- This step examines the raw columns for unusual characters, inconsistent formatting, or placeholder values that may affect downstream processing.  
- It helps identify early data-quality issues before performing IP parsing, feature engineering, or graph-based linking.




#### **2.1 Identify actual rows containing unusual characters in each column**

In [7]:
# 2.1 Identify actual rows containing unusual characters in each column 
def find_unusual_rows(series, allowed_pattern=r"[A-Za-z0-9\s\.\,\-\,(\)\/_]+"):
    unusual_mask = series.astype(str).apply(
        lambda x: not bool(re.fullmatch(allowed_pattern, x))
    )
    return series[unusual_mask]

for col in df.columns:
    print(f"\n=== Unusual values in column: {col} ===")
    unusual = find_unusual_rows(df[col])
    if len(unusual) == 0:
        print("No unusual values found.")
    else:
        display(unusual)


=== Unusual values in column: device_id ===
No unusual values found.

=== Unusual values in column: identity ===
No unusual values found.

=== Unusual values in column: bank ===
No unusual values found.

=== Unusual values in column: device_fingerprint ===
No unusual values found.

=== Unusual values in column: gpu_renderers ===
No unusual values found.

=== Unusual values in column: screen ===
No unusual values found.

=== Unusual values in column: os ===
No unusual values found.

=== Unusual values in column: browser ===
No unusual values found.

=== Unusual values in column: ips ===
No unusual values found.


**Insights**

- No columns returned values outside the allowed character set.  
- This indicates that all fields use consistent and structurally valid formats (UUIDs, hashes, user-agent strings, and IP lists) with no corrupted or malformed entries detected at the character level*. 


-------------------------------------------------

#### **2.2 Placeholder / Fake-missing values detection**

In [8]:
# 2.2 Placeholder / Fake-missing values detection ===

placeholders = ["-", "_", "", " ", "None", "none", "NULL", "null",
                "N/A", "n/a", "No data", "no data"]

for col in df.columns:
    mask = df[col].astype(str).str.strip().isin(placeholders)
    placeholder_values = df[col][mask].unique()
    
    print(f"\n=== Placeholder values in column: {col} ===")
    if len(placeholder_values) == 0:
        print("No placeholder values.")
    else:
        print(list(placeholder_values))




=== Placeholder values in column: device_id ===
No placeholder values.

=== Placeholder values in column: identity ===
['-']

=== Placeholder values in column: bank ===
No placeholder values.

=== Placeholder values in column: device_fingerprint ===
['No data']

=== Placeholder values in column: gpu_renderers ===
['-']

=== Placeholder values in column: screen ===
No placeholder values.

=== Placeholder values in column: os ===
No placeholder values.

=== Placeholder values in column: browser ===
No placeholder values.

=== Placeholder values in column: ips ===
No placeholder values.


**Insights**
- A scan for placeholder or fake-missing values (e.g., "-", "_", "No data") shows that only a very small subset of fields contains such entries.  
- These values represent non-informative placeholders rather than actual nulls, and appear mainly in identifier-related fields.  
- The presence of these placeholders is minimal and does not indicate structural data corruption.  
- They will be handled appropriately during the normalization and enrichment steps where required, while preserving any identifier fields that act as unique keys.
.*



------------------------

## **3) Data Engineering**

### **3.1 Normalize & Prepare Entities**
- We normalize identity, device_id, and device_fingerprint by converting them into 
sequential numeric codes (ID_1, ID_2, Dev_1, Dev_2, FP_1, FP_2). 
- This helps simplify link analysis and makes repeated entities easier to detect.


In [9]:
#Make a copy from the originel dataset
df1=df.copy()
# Ensure all entity fields are strings before encoding
df1['identity'] = df1['identity'].astype(str)
df1['device_id'] = df1['device_id'].astype(str)
df1['device_fingerprint'] = df1['device_fingerprint'].astype(str)


#### **3.1.1 Locate Original identities & device_id ip exploded**

In [10]:
# ---------------------------------------------------------
# Locate and document original device_id and identity values
# ---------------------------------------------------------

target_device = "91b12379-8098-457f-a2ad-a94d767797c2"
target_identity = "0007f265568f1abc1da791e852877df2047b3af9"

print(">>> Checking presence and row positions BEFORE encoding:\n")

# Locate rows for the device_id
device_rows = df1[df1["device_id"] == target_device]
print("Original device_id found in rows:")
display(device_rows)

# Locate rows for the identity
identity_rows = df1[df1["identity"] == target_identity]
print("Original identity found in rows:")
display(identity_rows)


>>> Checking presence and row positions BEFORE encoding:

Original device_id found in rows:


,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os,browser,ips
1219,91b12379-8098-457f-a2ad-a94d767797c2,00064120f0aa15e8c4197cf9f18a03a6e4bd35cb,Bank4,No data,-,"(736, 414, 32)",iOS 11.2.2,Mobile Safari UI/WKWebView 11.2.2,85.140.1.243
1220,91b12379-8098-457f-a2ad-a94d767797c2,-,Bank4,No data,-,"(736, 414, 32)",iOS 11.2.2,Mobile Safari UI/WKWebView 11.2.2,85.140.2.62
1221,91b12379-8098-457f-a2ad-a94d767797c2,-,Bank4,No data,-,"(736, 414, 32)",iOS 11.2.2,Mobile Safari UI/WKWebView 11.2.2,85.140.1.243


Original identity found in rows:


,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os,browser,ips
725,10645de7-f771-4b97-8dc4-c2de4d3cd9ec,0007f265568f1abc1da791e852877df2047b3af9,Bank8,No data,-,"(780, 360, 24)",Android,Chrome Mobile WebView 77.0.3865,213.234.222.26
726,10645de7-f771-4b97-8dc4-c2de4d3cd9ec,0007f265568f1abc1da791e852877df2047b3af9,Bank8,No data,-,"(780, 360, 24)",Android,Chrome Mobile WebView 77.0.3865,213.234.222.27
759,5d2c2da3-0d53-4444-a625-2480606e2c29,0007f265568f1abc1da791e852877df2047b3af9,Bank11,No data,-,"(720, 360, 24)",Android 8.1.0,Chrome Mobile WebView 78.0.3904,188.170.82.15
761,5d2c2da3-0d53-4444-a625-2480606e2c29,0007f265568f1abc1da791e852877df2047b3af9,Bank11,No data,-,"(720, 360, 24)",Android 8.1.0,Chrome Mobile WebView 78.0.3904,95.183.67.120
765,5d2c2da3-0d53-4444-a625-2480606e2c29,0007f265568f1abc1da791e852877df2047b3af9,Bank11,No data,-,"(720, 360, 24)",Android 8.1.0,Chrome Mobile WebView 78.0.3904,213.87.132.124
...,...,...,...,...,...,...,...,...,...
1375,cdaa8673-164d-44a7-80a0-99cd4cfe3818,0007f265568f1abc1da791e852877df2047b3af9,Bank11,69DBFA5B34C4E9919A68738F94D85DD59F35E968,-,"(720, 360, 24)",Android 8.1.0,Chrome Mobile WebView 79.0.3945,"213.87.157.236,188.170.82.31"
1376,cdaa8673-164d-44a7-80a0-99cd4cfe3818,0007f265568f1abc1da791e852877df2047b3af9,Bank11,69DBFA5B34C4E9919A68738F94D85DD59F35E968,-,"(720, 360, 24)",Android 8.1.0,Chrome Mobile WebView 79.0.3945,213.87.131.216
1377,cdaa8673-164d-44a7-80a0-99cd4cfe3818,0007f265568f1abc1da791e852877df2047b3af9,Bank11,69DBFA5B34C4E9919A68738F94D85DD59F35E968,-,"(720, 360, 24)",Android 8.1.0,Chrome Mobile WebView 79.0.3945,"213.87.147.83,213.87.149.20,213.87.132.170"
1378,cdaa8673-164d-44a7-80a0-99cd4cfe3818,0007f265568f1abc1da791e852877df2047b3af9,Bank11,69DBFA5B34C4E9919A68738F94D85DD59F35E968,-,"(720, 360, 24)",Android 8.1.0,Chrome Mobile WebView 79.0.3945,213.87.147.25


In [11]:
df1.head()

,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os,browser,ips
0,00993a56-f44b-448d-9d33-78c16f0be636,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac OS X 10.15.1,Chrome 78.0.3904,185.97.201.89
1,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"64.52.83.86,154.160.9.113,64.52.83.10"
2,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,154.160.10.201
3,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"64.52.83.148,64.52.83.232,154.160.9.113"
4,07e5a1d3-572d-4d5c-b7df-f903c9533fa7,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac OS X 10.15.1,Chrome 78.0.3904,217.66.159.132


### **3.1.2 Normalize & Expand IP Addresses**
- We clean and normalize the IP field,
- remove spaces, split multi-IP rows,
- and create an expanded IP table that supports graph-based link analysis.


#### **3.1.2.1 Clean IP Column** 

In [12]:
# Clean & Replace IP Column in df1
# ============================

def clean_ip_cell(cell):
    """
    Clean IP strings by removing spaces, dropping blanks,
    and removing duplicates inside each row.
    """
    if pd.isna(cell):
        return ""

    # remove spaces
    cell = str(cell).replace(" ", "")

    # split by comma
    ip_list = cell.split(",")

    # remove empty values
    ip_list = [ip.strip() for ip in ip_list if ip.strip() != ""]

    # remove duplicates in same row
    ip_list = list(dict.fromkeys(ip_list))

    # join back to one string
    return ",".join(ip_list)


# Apply cleaning
df1['ips'] = df1['ips'].apply(clean_ip_cell)

# Preview after replacement
display(df1.head())


,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os,browser,ips
0,00993a56-f44b-448d-9d33-78c16f0be636,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac OS X 10.15.1,Chrome 78.0.3904,185.97.201.89
1,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"64.52.83.86,154.160.9.113,64.52.83.10"
2,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,154.160.10.201
3,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"64.52.83.148,64.52.83.232,154.160.9.113"
4,07e5a1d3-572d-4d5c-b7df-f903c9533fa7,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac OS X 10.15.1,Chrome 78.0.3904,217.66.159.132


#### **3.1.2.2 Explode the IP column & All dataset** 

In [13]:
# Explode the IP column properly
df1['ips'] = df1['ips'].astype(str)              # ensure string format
df1['ips'] = df1['ips'].str.replace(" ", "")     # remove spaces
df1['ips'] = df1['ips'].str.split(',')           # convert to list
df1_exploded = df1.explode('ips').reset_index(drop=True)   # explode into multiple rows
df1_exploded = df1_exploded[df1_exploded['ips'].notna() & (df1_exploded['ips'] != "")]
display(df1_exploded.head())
display(f"The Shape of the new dataset: {df1_exploded.shape}")


,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os,browser,ips
0,00993a56-f44b-448d-9d33-78c16f0be636,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac OS X 10.15.1,Chrome 78.0.3904,185.97.201.89
1,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,64.52.83.86
2,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,154.160.9.113
3,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,64.52.83.10
4,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,154.160.10.201


'The Shape of the new dataset: (1802, 9)'

In [14]:
#Save the new dataset in csv file
df1_exploded.to_csv(r'C:\Users\Maha\OneDrive - Faculty Of Business (Ain Shams University)\Documents\machine learning og my theise\Final Analysis\Cyber Fraud Analyst_test_assignment\df_final.csv', index=False, encoding='utf-8-sig')

**Insight**
- Multiple IP addresses were stored in a single cell as comma-separated values.  
- This step cleans the field, splits all IPs into individual rows, and normalizes the dataset.  
- Exploding the IP column is essential to correctly measure IP usage, detect shared IP patterns,  
and identify suspicious connections across devices and identities.
- Save this changes in new csv file called df_final



#### **3.2.1.3 Geo-IP Enrichment (Using MaxMind GeoLite2)**

- The table below shows the enriched attributes extracted from GeoLite2 for each IP 
address, including country, city, subdivision, timezone, latitude/longitude, ASN, 
and provider. 
- These attributes help validate geographical consistency and detect 
anomalies such as VPN usage, mismatched locations, or suspicious hosting providers.




In [15]:
# Load GeoLite2 databases for City-level and ASN-level IP enrichment
city_reader = geoip2.database.Reader("GeoLite2-City.mmdb")
asn_reader = geoip2.database.Reader("GeoLite2-ASN.mmdb")

def geo_enrich(ip):
    try:
        # --- CITY LOOKUP ---
        # Retrieve geographical details from the City database
        city_res = city_reader.city(ip)

        # Extract country name
        country = city_res.country.name

        # Extract city name (may be None for some IP ranges)
        city = city_res.city.name

        # Extract most specific subdivision (region/state) if available
        subdivision = (
            city_res.subdivisions.most_specific.name
            if city_res.subdivisions
            else None
        )

        # Extract timezone information
        timezone = city_res.location.time_zone

        # Extract geographic coordinates (latitude & longitude)
        latitude = city_res.location.latitude
        longitude = city_res.location.longitude

    except:
        # In case the IP lookup fails → assign missing values
        country = city = subdivision = timezone = latitude = longitude = None

    try:
        # --- ASN LOOKUP ---
        # Retrieve Autonomous System details (network-level info)
        asn_res = asn_reader.asn(ip)

        # Extract ASN number (identifies the network owner)
        asn = asn_res.autonomous_system_number

        # Extract provider / organization name (ISP or hosting company)
        provider = asn_res.autonomous_system_organization

    except:
        # Fallback if ASN data is unavailable
        asn = provider = None

    # Return all extracted fields as a pandas Series
    return pd.Series({
        "country": country,
        "city": city,
        "subdivision": subdivision,
        "timezone": timezone,
        "latitude": latitude,
        "longitude": longitude,
        "asn": asn,
        "provider": provider
    })

# Apply Geo-IP enrichment to each exploded IP value
geo_df = df1_exploded["ips"].apply(geo_enrich)

# Merge enriched fields back into the main dataset
df3 = pd.concat([df1_exploded, geo_df], axis=1)

# Display first enriched records
display(df3.head())

# Save the enriched dataset to CSV for further analysis and reporting
output_csv_path = r'C:\Users\Maha\OneDrive - Faculty Of Business (Ain Shams University)\Documents\machine learning og my theise\Final Analysis\Cyber Fraud Analyst_test_assignment\df_final2.csv'
df3.to_csv(output_csv_path, index=True)

print(f"\nData successfully saved to: {output_csv_path}/n")
#Display qick info for detecting if there are missing values
display(df3.info())


,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os,browser,ips,country,city,subdivision,timezone,latitude,longitude,asn,provider
0,00993a56-f44b-448d-9d33-78c16f0be636,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac OS X 10.15.1,Chrome 78.0.3904,185.97.201.89,Russia,St Petersburg,St.-Petersburg,Europe/Moscow,59.9417,30.3096,39087.0,P.a.k.t LLC
1,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,64.52.83.86,United States,None,None,America/Chicago,37.7510,-97.8220,12182.0,INTERNAP-2BLK
2,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,154.160.9.113,Ghana,Accra,Greater Accra Region,Africa/Accra,5.5545,-0.1902,30986.0,SCANCOM
3,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,64.52.83.10,United States,None,None,America/Chicago,37.7510,-97.8220,12182.0,INTERNAP-2BLK
4,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,154.160.10.201,Ghana,Accra,Greater Accra Region,Africa/Accra,5.5545,-0.1902,30986.0,SCANCOM



Data successfully saved to: C:\Users\Maha\OneDrive - Faculty Of Business (Ain Shams University)\Documents\machine learning og my theise\Final Analysis\Cyber Fraud Analyst_test_assignment\df_final2.csv/n
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1802 entries, 0 to 1801
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   device_id           1802 non-null   object 
 1   identity            1802 non-null   object 
 2   bank                1802 non-null   object 
 3   device_fingerprint  1802 non-null   object 
 4   gpu_renderers       1802 non-null   object 
 5   screen              1802 non-null   object 
 6   os                  1802 non-null   object 
 7   browser             1802 non-null   object 
 8   ips                 1802 non-null   object 
 9   country             1802 non-null   object 
 10  city                1211 non-null   object 
 11  subdivision         1512 non-null   object 
 12

None

**Insight:**  
- Most IP addresses were successfully enriched with country, timezone, and provider 
information.
- Missing values in city or subdivision are expected for certain IP 
ranges (corporate networks, VPNs, or cloud infrastructure).  
- These enriched fields provide a solid foundation for later fraud-pattern detection, 
especially when cross-checking user locations versus device and bank activity.


--------------------------------------------------


### **3.1.3 "Device & Browser Attribute Enrichment (from User-Agent Components)"**
- This block cleans and standardizes the raw OS and Browser fields.
- What we do here:
- 1) Parse OS information → extract os_name, os_version, and infer device_type (Mobile/Desktop).
- 2) Parse Browser string → extract browser_name and browser_version.
-  3) Replace the original unstructured fields with these normalized attributes to ensure
 consistent, analytics-ready device features for later fraud-pattern detection.


In [16]:
# 1) Parse OS Information
#    Extracts OS name, version, and infers Mobile vs Desktop.
# -----------------------------------------------------------
def parse_os(os_str):
    if pd.isna(os_str):
        return pd.Series({"os_name": None, "os_version": None, "device_type": None})

    os_str = str(os_str).strip()
    parts = os_str.split()

    # OS name = first token
    os_name = parts[0]

    # OS version = everything after OS name
    os_version = " ".join(parts[1:]) if len(parts) > 1 else None

    # Infer device type from OS string
    lower_os = os_str.lower()
    if any(x in lower_os for x in ["android", "ios", "iphone", "ipad", "windows phone"]):
        device_type = "Mobile"
    else:
        device_type = "Desktop"

    return pd.Series({
        "os_name": os_name,
        "os_version": os_version,
        "device_type": device_type
    })


# -----------------------------------------------------------
# 2) Parse Browser Information 
#    Detects actual browser name even if "Mobile" or "iOS"
#    appears before it. Extracts version using regex.
# -----------------------------------------------------------
def parse_browser(br_str):
    if pd.isna(br_str):
        return pd.Series({"browser_name": None, "browser_version": None})

    br_str_original = str(br_str).strip()
    br_str = br_str_original.lower()

    # Identify browser name from anywhere in the string
    if "safari" in br_str and "chrome" not in br_str:
        browser_name = "Safari"
    elif "chrome" in br_str:
        browser_name = "Chrome"
    elif "firefox" in br_str:
        browser_name = "Firefox"
    elif "edge" in br_str:
        browser_name = "Edge"
    else:
        browser_name = "Other"

    # Extract browser version (first numeric pattern)
    match = re.search(r'\d+(\.\d+)*', br_str)
    browser_version = match.group(0) if match else None

    return pd.Series({
        "browser_name": browser_name,
        "browser_version": browser_version
    })


# -----------------------------------------------------------
# 3) Estimate Device Generation (for Mobile Devices Only)
#    Maps OS major version to approximate device age category.
# -----------------------------------------------------------
def mobile_generation(os_name, os_version):
    if os_name is None or os_version is None:
        return None

    os_name = str(os_name).lower()

    # Extract major version safely
    try:
        major_version = int(os_version.split('.')[0])
    except:
        return None

    # Android mapping
    if "android" in os_name:
        if major_version <= 7:
            return "Very Old Android Device"
        elif major_version == 8:
            return "Android 2017 Generation"
        elif major_version == 9:
            return "Android 2018 Generation"
        elif major_version == 10:
            return "Android 2019 Generation"
        elif major_version == 11:
            return "Android 2020 Generation"
        elif major_version == 12:
            return "Android 2021–2022 Generation"
        elif major_version == 13:
            return "Android 2023 Generation"
        else:
            return "Unknown_android"

    # iOS / iPhone mapping
    if "ios" in os_name or "iphone" in os_name:
        if major_version <= 12:
            return "Older iPhone Model"
        elif major_version == 13:
            return "iPhone 11 Generation"
        elif major_version == 14:
            return "iPhone 12 Generation"
        elif major_version == 15:
            return "iPhone 13 Generation"
        elif major_version == 16:
            return "iPhone 14 Generation"
        elif major_version == 17:
            return "iPhone 15 Generation"
        else:
            return "New iPhone Generation"

    return "Unknown_ios"


# -----------------------------------------------------------
# 4) Apply all parsing functions to the enriched dataset (df3)
# -----------------------------------------------------------
os_df = df3["os"].apply(parse_os)
browser_df = df3["browser"].apply(parse_browser)

# Add device generation column
os_df["device_generation"] = os_df.apply(
    lambda row: mobile_generation(row["os_name"], row["os_version"]),
    axis=1
)

# Merge parsed results back into dataset
df_useragent = pd.concat([df3, os_df, browser_df], axis=1)

# Preview
display(df_useragent.head())


,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os,browser,ips,country,city,subdivision,timezone,latitude,longitude,asn,provider,os_name,os_version,device_type,device_generation,browser_name,browser_version
0,00993a56-f44b-448d-9d33-78c16f0be636,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac OS X 10.15.1,Chrome 78.0.3904,185.97.201.89,Russia,St Petersburg,St.-Petersburg,Europe/Moscow,59.9417,30.3096,39087.0,P.a.k.t LLC,Mac,OS X 10.15.1,Desktop,None,Chrome,78.0.3904
1,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,64.52.83.86,United States,None,None,America/Chicago,37.7510,-97.8220,12182.0,INTERNAP-2BLK,iOS,12.4.1,Mobile,Older iPhone Model,Safari,12.1.2
2,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,154.160.9.113,Ghana,Accra,Greater Accra Region,Africa/Accra,5.5545,-0.1902,30986.0,SCANCOM,iOS,12.4.1,Mobile,Older iPhone Model,Safari,12.1.2
3,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,64.52.83.10,United States,None,None,America/Chicago,37.7510,-97.8220,12182.0,INTERNAP-2BLK,iOS,12.4.1,Mobile,Older iPhone Model,Safari,12.1.2
4,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,154.160.10.201,Ghana,Accra,Greater Accra Region,Africa/Accra,5.5545,-0.1902,30986.0,SCANCOM,iOS,12.4.1,Mobile,Older iPhone Model,Safari,12.1.2


In [17]:
# -----------------------------------------------------------
# Drop the original raw OS and Browser fields since they have
# now been replaced by normalized, structured User-Agent features.
# -----------------------------------------------------------
df_clean = df_useragent.drop(columns=["os", "browser"])


# -----------------------------------------------------------
# Define the new extracted User-Agent feature columns that
# should be inserted directly after the 'screen' column.
# These fields provide clearer device and browser signals for
# fraud-pattern detection.
# -----------------------------------------------------------
new_cols = [
    "os_name",
    "os_version",
    "browser_name",
    "browser_version",
    "device_type",
    "device_generation"
]


# -----------------------------------------------------------
# Collect all existing columns except the newly added ones.
# This ensures we do not duplicate columns and that we preserve
# the original dataset schema as much as possible.
# -----------------------------------------------------------
original_cols = [c for c in df_clean.columns if c not in new_cols]


# -----------------------------------------------------------
# Locate the index of the 'screen' column so we can insert the
# new User-Agent attributes immediately after it.
# -----------------------------------------------------------
screen_index = original_cols.index("screen") + 1


# -----------------------------------------------------------
# Build the final column ordering:
#  - All original columns up to 'screen'
#  - Then all new User-Agent feature columns
#  - Then the remaining original columns
# This keeps the dataset tidy and logically structured.
# -----------------------------------------------------------
final_columns = (
    original_cols[:screen_index] +
    new_cols +
    original_cols[screen_index:]
)


# -----------------------------------------------------------
# Apply the new column ordering to create the final dataset
# with enriched device/browser attributes placed correctly.
# -----------------------------------------------------------
df_clean = df_clean[final_columns]


# -----------------------------------------------------------
# Display a preview of the final structured dataset to verify
# correct column placement and formatting.
# -----------------------------------------------------------
display(df_clean.head())


,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os_name,os_version,browser_name,browser_version,device_type,device_generation,ips,country,city,subdivision,timezone,latitude,longitude,asn,provider
0,00993a56-f44b-448d-9d33-78c16f0be636,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac,OS X 10.15.1,Chrome,78.0.3904,Desktop,None,185.97.201.89,Russia,St Petersburg,St.-Petersburg,Europe/Moscow,59.9417,30.3096,39087.0,P.a.k.t LLC
1,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS,12.4.1,Safari,12.1.2,Mobile,Older iPhone Model,64.52.83.86,United States,None,None,America/Chicago,37.7510,-97.8220,12182.0,INTERNAP-2BLK
2,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS,12.4.1,Safari,12.1.2,Mobile,Older iPhone Model,154.160.9.113,Ghana,Accra,Greater Accra Region,Africa/Accra,5.5545,-0.1902,30986.0,SCANCOM
3,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS,12.4.1,Safari,12.1.2,Mobile,Older iPhone Model,64.52.83.10,United States,None,None,America/Chicago,37.7510,-97.8220,12182.0,INTERNAP-2BLK
4,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS,12.4.1,Safari,12.1.2,Mobile,Older iPhone Model,154.160.10.201,Ghana,Accra,Greater Accra Region,Africa/Accra,5.5545,-0.1902,30986.0,SCANCOM


In [18]:
# Replace all missing values in the dataframe with "Unknown"
df_clean = df_clean.fillna("Unknown")

display(df_clean.head())
display(df_clean.isnull().count())
# --- Save the new enriched data to CSV ---
output_csv_path = r'C:\Users\Maha\OneDrive - Faculty Of Business (Ain Shams University)\Documents\machine learning og my theise\Final Analysis\Cyber Fraud Analyst_test_assignment\Assignment_docoments\df_final2.csv'
df_clean.to_csv(output_csv_path, index=False)
print(f"\nData successfully saved to: {output_csv_path}")

,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os_name,os_version,browser_name,browser_version,device_type,device_generation,ips,country,city,subdivision,timezone,latitude,longitude,asn,provider
0,00993a56-f44b-448d-9d33-78c16f0be636,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac,OS X 10.15.1,Chrome,78.0.3904,Desktop,Unknown,185.97.201.89,Russia,St Petersburg,St.-Petersburg,Europe/Moscow,59.9417,30.3096,39087.0,P.a.k.t LLC
1,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS,12.4.1,Safari,12.1.2,Mobile,Older iPhone Model,64.52.83.86,United States,Unknown,Unknown,America/Chicago,37.7510,-97.8220,12182.0,INTERNAP-2BLK
2,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS,12.4.1,Safari,12.1.2,Mobile,Older iPhone Model,154.160.9.113,Ghana,Accra,Greater Accra Region,Africa/Accra,5.5545,-0.1902,30986.0,SCANCOM
3,02591b70-e8f5-4299-901c-e78b2b79b526,-,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS,12.4.1,Safari,12.1.2,Mobile,Older iPhone Model,64.52.83.10,United States,Unknown,Unknown,America/Chicago,37.7510,-97.8220,12182.0,INTERNAP-2BLK
4,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS,12.4.1,Safari,12.1.2,Mobile,Older iPhone Model,154.160.10.201,Ghana,Accra,Greater Accra Region,Africa/Accra,5.5545,-0.1902,30986.0,SCANCOM


device_id             1802
identity              1802
bank                  1802
device_fingerprint    1802
gpu_renderers         1802
screen                1802
os_name               1802
os_version            1802
browser_name          1802
browser_version       1802
device_type           1802
device_generation     1802
ips                   1802
country               1802
city                  1802
subdivision           1802
timezone              1802
latitude              1802
longitude             1802
asn                   1802
provider              1802
dtype: int64


Data successfully saved to: C:\Users\Maha\OneDrive - Faculty Of Business (Ain Shams University)\Documents\machine learning og my theise\Final Analysis\Cyber Fraud Analyst_test_assignment\Assignment_docoments\df_final2.csv


**Summary of Device & Browser Enrichment Results**

After applying the User-Agent Normalization pipeline, the dataset now contains a fully structured and standardized set of device attributes.  
These transformations significantly improve the reliability of downstream fraud-pattern analysis.

##### 1) User-Agent Attribute Extraction  
The raw `os` and `browser` fields were converted into analytically usable features, including:
- **os_name** and **os_version**  
- **browser_name** and **browser_version**  
- **device_type** (Mobile vs Desktop)  
- **device_generation** (estimated device age category)

This normalization removes ambiguity from the original text fields and exposes clearer behavioral signals, especially for multi-device or automated activity patterns.

#### 2) Schema Refinement & Column Organization  
The original unstructured fields (`os`, `browser`) were removed and replaced with the richer parsed attributes.  
All new fields were inserted immediately after the `screen` column to maintain a logically grouped device-profiling section within the dataset.  
This ensures the table remains clean, readable, and aligned with forensic analysis practices.

#### 3) Missing-Value Treatment  
All remaining NULL values across the enriched dataset were replaced with **"Unknown"**.  
This prevents downstream analytical errors, ensures stable aggregations, and keeps the feature space consistent.  
The use of "Unknown" is intentional: it allows modeling and fraud-detection logic to treat missing device metadata as a distinct behavioral signal—not simply ignored data.

---

### Result  
The final dataset is now:
- Structurally consistent  
- Device-profile enriched  
- Fully analyzable without data interruptions  
- Better suited for detecting anomalous device/browser behavior, cross-device usage, or automated fraud indicators  

This enriched feature set forms the foundation for the next stages of fraud-pattern mapping and correlation analysis.
